# display version of python used 

In [1]:
!py --version

Python 3.10.8


# import the libary

In [2]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccac

# device used

In [3]:
print(paddle.device.get_device())

cpu


# loading image path

In [4]:
i=4
image_path1 = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path1

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/4.jpg'

# run paddleocr on img 

In [5]:
img = cv2.imread(image_path1)
print(img is None)
# setting tread 
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"

ocr = PaddleOCR(use_doc_orientation_classify=False, 
    use_doc_unwarping=False, 
    use_textline_orientation=False,
    lang='en',
    device='cpu',
    cpu_threads=8 
   )

result1 = ocr.predict(img)

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.


False


Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


In [6]:
result1

[{'input_path': None,
  'page_index': None,
  'doc_preprocessor_res': {'output_img': array([[[255, ..., 255],
           ...,
           [255, ..., 255]],
   
          ...,
   
          [[255, ..., 255],
           ...,
           [255, ..., 255]]], shape=(1580, 1998, 3), dtype=uint8)},
  'dt_polys': [array([[58,  4],
          ...,
          [56, 44]], shape=(4, 2), dtype=int16),
   array([[552,  19],
          ...,
          [551,  54]], shape=(4, 2), dtype=int16),
   array([[55, 41],
          ...,
          [55, 81]], shape=(4, 2), dtype=int16),
   array([[1304,   34],
          ...,
          [1303,   74]], shape=(4, 2), dtype=int16),
   array([[1497,   40],
          ...,
          [1496,   76]], shape=(4, 2), dtype=int16),
   array([[1628,   46],
          ...,
          [1628,   82]], shape=(4, 2), dtype=int16),
   array([[1754,   45],
          ...,
          [1753,   77]], shape=(4, 2), dtype=int16),
   array([[ 58,  99],
          ...,
          [ 58, 135]], shape=(4, 2), 

In [23]:
ocr_result = [(poly , text , scores)
    for poly , text , scores  in 
        zip(result1[0]['dt_polys'],
            result1[0]['rec_texts'],
            result1[0]['rec_scores'])]

In [25]:
ocr_result[0]

(array([[58,  4],
        ...,
        [56, 44]], shape=(4, 2), dtype=int16),
 'SI',
 0.9975608587265015)

In [49]:
result1[0]['dt_polys'][1].tolist()

[[552, 19], [819, 26], [818, 62], [551, 54]]

In [50]:
result1[0]['rec_polys'][1]

array([[552,  19],
       ...,
       [551,  54]], shape=(4, 2), dtype=int16)

In [65]:
def get_center(box):
    x_coords = [p[0] for p in box]
    y_coords = [p[1] for p in box]
    return sum(x_coords) / 4, sum(y_coords) / 4


def group_boxes_into_lines(boxes, y_threshold=20):
    # Step 1: compute centers
    box_centers = [(box, get_center(box)) for box in boxes]

    # Step 2: sort by Y (top to bottom)
    box_centers.sort(key=lambda b: b[1][1])

    lines = []
    current_line = []

    for box, (cx, cy) in box_centers:
        if not current_line:
            current_line.append((box, cx, cy))
            continue

        _, _, prev_cy = current_line[-1]

        # Step 3: check if same line
        if abs(cy - prev_cy) < y_threshold:
            current_line.append((box, cx, cy))
        else:
            lines.append(current_line)
            current_line = [(box, cx, cy)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line by X (left to right)
    sorted_lines = []
    for line in lines:
        line.sort(key=lambda b: b[1])  # sort by center x
        sorted_lines.append([b[0] for b in line])

    return sorted_lines

In [59]:
rs = group_boxes_into_lines(result1[0]['rec_polys'])
for k in range(len(result1[0]['rec_polys'])):
    for i in range(len(rs)):
        for j in range(len(rs[i])):
            # print(rs[i][j])
            # print(result1[0]['rec_polys'][k])
            # if(rs[i][j].all() == result1[0]['rec_polys'][k].all()):
            if np.array_equal(rs[i][j], result1[0]['rec_polys'][k]):
                print(result1[0]['rec_texts'][k])
    print("\n")
    print("-----------------------------")
    print("\n")

SI


-----------------------------


Description of Goods


-----------------------------


No.


-----------------------------


Quantity


-----------------------------


Rate


-----------------------------


per


-----------------------------


Amount


-----------------------------


1


-----------------------------


Books


-----------------------------


5,000 Nos


-----------------------------


Record Book


-----------------------------


38.00


-----------------------------


Nos


-----------------------------


1,90,000.00


-----------------------------


Size-1/4th Pages-4+96


-----------------------------


Cover-3o0gsm Duplex Board


-----------------------------


With Multicolor Printing


-----------------------------


& Matt Lamination


-----------------------------


Inner 60gsm Maplitho


-----------------------------


Single Color Printing


-----------------------------


Finishing-Perfect Binding


-----------------------------


2


-----------------

In [66]:
import numpy as np

grouped_texts = []

for line in rs:
    line_texts = []
    for box in line:
        # Find index of this box in original rec_polys
        for idx, orig_box in enumerate(result1[0]['rec_polys']):
            if np.array_equal(box, orig_box):
                line_texts.append(result1[0]['rec_texts'][idx])
                break
    grouped_texts.append(line_texts)

# Print grouped text lines
for line in grouped_texts:
    print(" ".join(line))

No. SI Description of Goods Quantity Rate per Amount
1 Books
Record Book 5,000 Nos 38.00 Nos 1,90,000.00
Size-1/4th Pages-4+96
Cover-3o0gsm Duplex Board
With Multicolor Printing
& Matt Lamination
Inner 60gsm Maplitho
Single Color Printing
Finishing-Perfect Binding
2 Books
Record Book 2,500 Nos 52.00 Nos 1,30,000.00
Size-1/4th Pages-4+144
Cover 300gsm Dup Board
With Multicolor Printing
& Matt Lamination
Inner-60gsm Maplitho Single
Color Printing
Finishing-Perfect Binding
3,20,000.00
Cow Bu 415342 CGST (Output) 9% SGST (Output) 9% 9 9 % % 28,800.00 28,800.00
ou10226
Amount Chargeable (in words) Total 7,500 Nos 3,77,600.00
E. &O.E
INR Three Lakh Seventy Seven Thousand Six Hundred Only
HSN/SAC Taxable Central Tax State Tax Total


In [53]:
 # cv2.polylines(output, [np.array(rs[0], dtype=np.int16)],isClosed=True, color=(0, 255, 255),thickness=4)
 # cv2.polylines(output, rs[0],isClosed=True, color=(0, 255, 255),thickness=4)
# cv2.drawContours(output, [np.array(rs[0], dtype=np.int16)], -1, (0,255,0), 4)

# marking the table or extracting the table lines

In [23]:
i=15
image_path1 = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpeg"
image_path1

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/15.jpeg'

In [28]:
img = cv2.imread(image_path1)
# img = sharpened
if img is None:
    print("Image not loaded. Check path!")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# cv2.imwrite('./output/opencv/gray1.jpeg', gray)


clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
gray = clahe.apply(gray)
# cv2.imwrite('./output/opencv/gray2.jpeg', gray)



# ✅ Better threshold (adaptive handles uneven lighting)
thresh = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    15, 2
)



kernel = np.zeros((2,2), np.uint8)
thresh = cv2.erode(thresh, kernel, iterations=2)
cv2.imwrite('./output/opencv/thresh1.jpeg', thresh)


# --- Horizontal lines ---
horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
cv2.imwrite('./output/opencv/horizontal.jpeg', horizontal)

# 🔥 connect broken horizontal lines
horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=2)
cv2.imwrite('./output/opencv/horizontal1.jpeg', horizontal)

# --- Vertical lines ---
vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
# cv2.imwrite('./output/opencv/vertical.jpeg', vertical)

# 🔥 connect broken vertical lines
vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=2)
# cv2.imwrite('./output/opencv/vertical1.jpeg', vertical)


# Combine
boxes = cv2.add(horizontal, vertical)
# 1. Dilate both masks slightly to ensure overlap
# kernel_overlap = np.ones((3,3), np.uint8)
# dilated_h = cv2.dilate(horizontal, kernel_overlap, iterations=2)
# dilated_v = cv2.dilate(vertical, kernel_overlap, iterations=2)

# # 2. Now find the intersection
# boxes = cv2.bitwise_and(dilated_h, dilated_v)

# 3. (Optional) Filter these dots to find the exact corners
# These dots represent the 'joints' of your table
cv2.imwrite(f'./output/opencv/boxes{i}.jpeg', boxes)

# 🔥 Final closing to join gaps
kernel = np.ones((10,10), np.uint8)
boxes = cv2.morphologyEx(boxes, cv2.MORPH_CLOSE, kernel, iterations=2)
cv2.imwrite(f'./output/opencv/boxes{i}.jpeg', boxes)



# kernel = np.ones((3,3), np.uint8)
# Apply erosion
# eroded = cv2.erode(boxes, kernel, iterations=1)
# cv2.imwrite('./output/opencv/boxes2.jpeg', eroded)

# Find contours
contours, _ = cv2.findContours(boxes, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
# cv2.imwrite('./output/opencv/contours.jpeg', contours)

output = img.copy()
image_out = cv2.imread('./output/opencv/boxes1.jpeg')

# for index,cnt in enumerate(contours):
#     x,y,w,h = cv2.boundingRect(cnt)
#     # filter noise
#     # if w > 100 and h > 100:
cv2.drawContours(output, contours, -1, (0,255,0), 1)
cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)

True

In [26]:
import cv2

def get_center_from_rect(rect):
    x, y, w, h = rect
    cx = x + w / 2
    cy = y + h / 2
    return cx, cy


def group_contours_into_lines(contours, y_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [cv2.boundingRect(cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, *get_center_from_rect(rect)) for rect in rects]
    #[[x1,y1,w1,h1],cx,yc]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[2])  # sort by cy

    lines = []
    current_line = []

    for rect, cx, cy in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy))
            continue
            
        #get the last cy
        _, _, prev_cy = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cy - prev_cy) < y_threshold:
            current_line.append((rect, cx, cy))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    for line in lines:
        line.sort(key=lambda r: r[1])  # sort by cx
        result.append([rect for rect, _, _ in line])

    return result

In [29]:
ans = group_contours_into_lines(contours)

In [31]:
ans

[[(604, 5, 525, 34), (1132, 5, 810, 48)],
 [(61, 15, 46, 72),
  (110, 17, 1184, 99),
  (1297, 47, 163, 72),
  (1462, 51, 165, 71),
  (1629, 54, 71, 69),
  (1702, 54, 239, 71)],
 [(1687, 127, 252, 684)],
 [(46, 88, 60, 1325),
  (94, 90, 1199, 1334),
  (0, 0, 1998, 1580),
  (1275, 121, 184, 1305),
  (1440, 123, 185, 1307),
  (1606, 126, 93, 1305)],
 [(1679, 811, 244, 626)],
 [(44, 1415, 47, 36),
  (93, 1416, 1179, 45),
  (1275, 1427, 162, 37),
  (1439, 1431, 164, 37),
  (1606, 1434, 70, 35),
  (1678, 1436, 238, 39)],
 [(43, 1453, 1872, 110)]]

In [21]:
contours[2][:, 0, :].tolist()

[[1754, 1375],
 [1755, 1374],
 [1801, 1374],
 [1802, 1375],
 [1903, 1375],
 [1904, 1376],
 [2012, 1376],
 [2013, 1377],
 [2013, 1397],
 [2014, 1398],
 [2014, 1420],
 [2013, 1421],
 [1991, 1421],
 [1990, 1420],
 [1943, 1420],
 [1942, 1419],
 [1856, 1419],
 [1855, 1418],
 [1755, 1418],
 [1754, 1417]]

# scaling the paddle ocr border to small

In [ ]:
new_poly = []
for poly in result1[0]["rec_polys"]:
    # convert to float for calculation
    poly = np.array(poly, dtype=np.float32)

    x_min, y_min = np.min(poly, axis=0)
    x_max, y_max = np.max(poly, axis=0)
    w = x_max - x_min
    h = y_max - y_min

    # adaptive shrink
    shrink_factor_h = max(0.6, min(0.9, 1 - 20/w))
    shrink_factor_v = max(0.6, min(0.9, 1 - 20/h))
    # print(poly)
    
    # compute center
    center_x = np.mean(poly[:, 0])
    center_y = np.mean(poly[:, 1])

    poly_box=[]
    for (x,y) in poly:
        #x' = Xc + (X-Xc)*srink
        x_new=center_x+(x-center_x)*shrink_factor_h
        y_new=center_y+(y-center_y)*shrink_factor_v
        poly_box.append([x_new,y_new])
    new_poly.append(poly_box)
cv2.polylines(output, [np.array(poly_box, dtype=np.int32)],isClosed=True, color=(0, 255, 255),thickness=1)    
cv2.imwrite('./output/opencv/imagetry.jpeg', output)

# trying to get the overlapping

In [22]:
import numpy as np

def rectangle_overlap_percentage(rectA, rectB):
    """
    rectA and rectB: list of 4 points [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    Can also be numpy arrays of shape (4,2)
    Returns overlap % relative to rectA
    """
    # Convert to numpy arrays
    rectA = np.array(rectA)
    rectB = np.array(rectB)
    
    # Get bounding box
    x_min_A, x_max_A = rectA[:,0].min(), rectA[:,0].max()
    y_min_A, y_max_A = rectA[:,1].min(), rectA[:,1].max()
    
    x_min_B, x_max_B = rectB[:,0].min(), rectB[:,0].max()
    y_min_B, y_max_B = rectB[:,1].min(), rectB[:,1].max()
    
    # Overlap width & height
    overlap_width = max(0, min(x_max_A, x_max_B) - max(x_min_A, x_min_B))
    overlap_height = max(0, min(y_max_A, y_max_B) - max(y_min_A, y_min_B))
    
    # Intersection area
    intersection_area = overlap_width * overlap_height
    
    # Area of rectangle A
    area_A = (x_max_A - x_min_A) * (y_max_A - y_min_A)
    
    # Overlap %
    overlap_percentage = (intersection_area / area_A) * 100 if area_A > 0 else 0
    
    return overlap_percentage


# # Example usage:
# rectA = np.array([[1, 1], [1, 4], [5, 4], [5, 1]])
# rectB = np.array([[3, 2], [3, 5], [6, 5], [6, 2]])

# overlap = rectangle_overlap_percentage(rectA, rectB)
# print(f"Overlap % of Rectangle A: {overlap:.2f}%")


In [23]:
for i in range(0,len(result1[0]['rec_polys'])):
    for j in range(0, len(contours)):
        overlap = rectangle_overlap_percentage(result1[0]['rec_polys'][i].tolist(),contours[j][:, 0, :].tolist())
        if(overlap > 50):
            print(f"Overlap % of Rectangle A: {overlap:.2f}%")
            cv2.drawContours(output, contours, j, (255,255,0), 2)
            cv2.polylines(output, [np.array(result1[0]['rec_polys'][i], dtype=np.int32)],isClosed=True, color=(0, 0, 255),thickness=2)
cv2.imwrite('./output/opencv/imagetry.jpeg', output)

Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 69.98%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 84.62%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 72.97%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 75.68%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 75.56%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 76.47%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 75.00%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 85.00%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 80.56%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 90.20%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 98.17%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 99.83%
Overlap % of Rectangle A: 100.00%
Overlap % of Rectangle A: 

True

In [82]:
result1[0]['rec_polys'][0]

array([[30,  6],
       ...,
       [27, 46]], shape=(4, 2), dtype=int16)

# unused or unwanted code 

In [8]:
h_groups, v_groups = group_ocr_results(result1[0]['rec_polys'])

# print horizontal lines
for row in h_groups:
    line = " ".join([item[1][0] for item in row["items"]])
    print("H:", line)

# print vertical columns
for col in v_groups:
    line = " | ".join([item[1][0] for item in col["items"]])
    print("V:", line)

ValueError: too many values to unpack (expected 2)

In [8]:
stdTextBool=False
for index,text in enumerate(result1[0]['rec_texts']):
    stdText=text.strip()
    stdText=' '.join(stdText.split())
    stdText=stdText.lower()
    # print(stdText)
    tableHead=[]
    if(stdText == "no" or stdText=="no." ):
        stdTextBool=False
    elif(stdText == "si" or stdTextBool==True):
        stdTextBool=True
        tableHead.append(stdText)
        print(text)
    

SI
No. & Kind
Descriplion of Goods
HSN/SAC
Quanlity
Rale
per
Disc
%
Amount


In [5]:
import numpy as np
from collections import defaultdict

def get_bbox(poly):
    xs = [p[0] for p in poly]
    ys = [p[1] for p in poly]
    return min(xs), min(ys), max(xs), max(ys)

def get_orientation(poly):
    # width vs height
    x_min, y_min, x_max, y_max = get_bbox(poly)
    w = x_max - x_min
    h = y_max - y_min
    
    if w >= h:
        return "horizontal"
    else:
        return "vertical"

def group_horizontal(boxes, y_thresh=15):
    """
    Group boxes into rows (horizontal lines)
    """
    rows = []
    
    for box in boxes:
        poly, (text, conf) = box
        _, y_min, _, y_max = get_bbox(poly)
        center_y = (y_min + y_max) / 2
        
        placed = False
        for row in rows:
            if abs(row["center_y"] - center_y) < y_thresh:
                row["items"].append(box)
                row["center_y"] = (row["center_y"] + center_y) / 2
                placed = True
                break
        
        if not placed:
            rows.append({
                "center_y": center_y,
                "items": [box]
            })
    
    # sort each row left → right
    for row in rows:
        row["items"].sort(key=lambda b: get_bbox(b[0])[0])
    
    return rows

def group_vertical(boxes, x_thresh=15):
    """
    Group boxes into columns (vertical lines)
    """
    cols = []
    
    for box in boxes:
        poly, (text, conf) = box
        x_min, _, x_max, _ = get_bbox(poly)
        center_x = (x_min + x_max) / 2
        
        placed = False
        for col in cols:
            if abs(col["center_x"] - center_x) < x_thresh:
                col["items"].append(box)
                col["center_x"] = (col["center_x"] + center_x) / 2
                placed = True
                break
        
        if not placed:
            cols.append({
                "center_x": center_x,
                "items": [box]
            })
    
    # sort each column top → bottom
    for col in cols:
        col["items"].sort(key=lambda b: get_bbox(b[0])[1])
    
    return cols

def group_ocr_results(results):
    horizontal_boxes = []
    vertical_boxes = []
    
    for res in results:
        poly, (text, conf) = res
        orientation = get_orientation(poly)
        
        if orientation == "horizontal":
            horizontal_boxes.append(res)
        else:
            vertical_boxes.append(res)
    
    h_groups = group_horizontal(horizontal_boxes)
    v_groups = group_vertical(vertical_boxes)
    
    return h_groups, v_groups

In [28]:
# # import cv2
# import numpy as np

# # Load image (COLOR, not grayscale)
# img = cv2.imread(image_path1)

# # Step 1: White balance (Gray World assumption)
# result = img.copy().astype(np.float32)

# avg_b = np.mean(result[:,:,0])
# avg_g = np.mean(result[:,:,1])
# avg_r = np.mean(result[:,:,2])

# avg_gray = (avg_b + avg_g + avg_r) / 3

# result[:,:,0] *= (avg_gray / avg_b)
# result[:,:,1] *= (avg_gray / avg_g)
# result[:,:,2] *= (avg_gray / avg_r)

# result = np.clip(result, 0, 255).astype(np.uint8)

# # Step 2: Contrast enhancement (CLAHE on LAB space)
# lab = cv2.cvtColor(result, cv2.COLOR_BGR2LAB)
# l, a, b = cv2.split(lab)

# clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
# l = clahe.apply(l)

# lab = cv2.merge((l, a, b))
# enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

# # Step 3: Denoising (mild)
# denoised = cv2.fastNlMeansDenoisingColored(enhanced, None, 10, 10, 7, 21)

# # Step 4: Sharpening
# kernel = np.array([[0, -1, 0],
#                    [-1, 5,-1],
#                    [0, -1, 0]])

# sharpened = cv2.filter2D(denoised, -1, kernel)

# # Save result
# cv2.imwrite("./output/opencv/auto_color_result.jpg", sharpened)

True

In [ ]:
# 1. Combine your raw horizontal and vertical lines
combined_lines = cv2.add(horizontal, vertical)

# 2. Use a small Rectangular Kernel for a "Clean Join"
# This kernel size (5x5) is enough to bridge 1-2 pixel gaps 
# without making the lines 10 pixels thick.
join_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

# 3. Apply MORPH_CLOSE
# This joins the 'broken' parts of the lines into a solid grid
clean_grid = cv2.morphologyEx(combined_lines, cv2.MORPH_CLOSE, join_kernel, iterations=1)